In [120]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [121]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')


# model
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    vertexai = True,
    project = project
)

In [122]:
# lets define basic tools
from langchain_core.tools import tool

# create and register tools

@tool
def hello(name:str) -> str:
    """Greets a person by name

    Args:
        name (str): The name of the person to greet

    Returns:
        str: A greeting message
    """
    return f"Hello, {name}!"

In [123]:
llm_with_tools = llm.bind_tools([hello])

In [124]:
response = llm.invoke("Greet Hero")
response

AIMessage(content="Hello Hero! It's an honor to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8f64-0563-7670-8de5-63df04d14d00-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 19, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}})

In [125]:
response = llm_with_tools.invoke("Greet Hero")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'hello', 'arguments': '{"name": "Hero"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8f64-09b8-7ac1-a1eb-291435bda27e-0', tool_calls=[{'name': 'hello', 'args': {'name': 'Hero'}, 'id': '0cbf01a0-483c-4866-b47b-385c540117ff', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 3, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}})

In [126]:
from langchain.agents import create_agent
agent = create_agent(model=llm,
                     tools=[hello])

In [127]:
agent_response=agent.invoke({
    "messages": [
        SystemMessage(content="You are a helpful assistant that greets people by name."),
        HumanMessage(content="Greet Hero")
    ]
})


In [128]:
agent_response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Hello, Hero!


In [129]:
# i want search internet
#os.getenv('TAVILY_API_KEY')
!uv add langchain_tavily

Resolved 86 packages in 2ms
Audited 83 packages in 0.95ms


In [130]:
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=3,
    topic="news"
)


In [131]:
#let's create agent with tavily search tool
agent = create_agent(model=llm,
                                tools=[hello, tavily_search_tool])


In [132]:
# response=agent.invoke({
#     "messages": "who won the ipl match on apr 15th 2026? and use tavily_search tool to find the answer"
#     }
#     )

# response=agent.invoke({
#     "messages": "what nation hosted the Euro 2024 football tournament?"
#     }
#     )

response=agent.invoke({
    "messages": "what is the latest news on ipl and who won yesderday that is apr 15?"
    }
    )

In [133]:
response['messages']

[HumanMessage(content='what is the latest news on ipl and who won yesderday that is apr 15?', additional_kwargs={}, response_metadata={}, id='aca200dc-36f8-46c8-9d58-81f30dcafdaa'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"end_date": "2024-04-15", "query": "who won IPL match on April 15 2024", "start_date": "2024-04-15"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8f64-133f-75e1-967b-f7a78e6350a2-0', tool_calls=[{'name': 'tavily_search', 'args': {'time_range': 'day', 'query': 'IPL news'}, 'id': '17887aab-97ae-4950-99ab-fc6a92effe6b', 'type': 'tool_call'}, {'name': 'tavily_search', 'args': {'end_date': '2024-04-15', 'query': 'who won IPL match on April 15 2024', 'start_date': '2024-04-15'}, 'id': '7ece0ec5-1e62-4c94-ac57-e042ac21c1bd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1

In [134]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

I'm sorry, but I cannot fulfill this request. The start and end dates for the search query cannot be the same.


In [135]:
!uv add yfinance langchain_community

Resolved 86 packages in 2ms
Audited 83 packages in 1ms


In [140]:
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

yahoo_finance_news_tool = YahooFinanceNewsTool()

In [141]:
fin_agent = create_agent(model=llm,
                         tools=[yahoo_finance_news_tool])


In [142]:
response = fin_agent.invoke({
    "messages": "what is news on Apple Inc. stock?"
})

ImportError: Could not import yfinance python package. Please install it with `pip install yfinance`.

In [ ]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

what is stock price of Apple Inc.?
================================== Ai Message ==================================

I cannot provide real-time stock prices. I can provide financial news about companies if you give me their ticker symbol.
